# 01 — Extração (E do ETL)

Este notebook extrai dados de **3 fontes heterogêneas**:
1. **CSV** — Transações financeiras pessoais (sintético, simulando dados reais)
2. **API REST** — Cotações históricas do Ibovespa via `yfinance`
3. **Excel** — Planilha de orçamento empresarial

Os dados brutos são salvos em formato **Parquet** em `data/raw/`.

In [ ]:
import os
import pandas as pd
import yfinance as yf

# Garante que as pastas existam
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Ambiente pronto.")

## 1.1 — Fonte CSV: Transações Financeiras

In [ ]:
df_csv = pd.read_csv("data/source/personal_finance.csv")

print(f"Registros: {len(df_csv)}")
print(f"Colunas:   {list(df_csv.columns)}")
df_csv.dtypes

In [ ]:
df_csv.head(10)

In [ ]:
df_csv.describe()

In [ ]:
df_csv.to_parquet("data/raw/finance_csv.parquet", index=False)
print("✔ Salvo em data/raw/finance_csv.parquet")

## 1.2 — Fonte API REST: Ibovespa (yfinance)

In [ ]:
df_ibov = yf.download("^BVSP", start="2022-01-01", end="2024-12-31", progress=False)

# yfinance retorna MultiIndex quando se pede 1 ticker; flatten
if isinstance(df_ibov.columns, pd.MultiIndex):
    df_ibov.columns = df_ibov.columns.get_level_values(0)

df_ibov = df_ibov.reset_index()
print(f"Registros Ibovespa: {len(df_ibov)} dias de pregão")
df_ibov.head()

In [ ]:
df_ibov.describe()

In [ ]:
df_ibov.to_parquet("data/raw/ibovespa.parquet", index=False)
print("✔ Salvo em data/raw/ibovespa.parquet")

## 1.3 — Fonte Excel: Orçamento Empresarial

In [ ]:
df_xl = pd.read_excel("data/source/orcamento.xlsx", sheet_name="Dados")

print(f"Registros orçamento: {len(df_xl)}")
print(f"Colunas: {list(df_xl.columns)}")
df_xl.head(10)

In [ ]:
df_xl.to_parquet("data/raw/orcamento.parquet", index=False)
print("✔ Salvo em data/raw/orcamento.parquet")

---
## Resumo da Extração

| Fonte | Formato | Registros | Destino |
|-------|---------|-----------|--------|
| Transações financeiras | CSV → Parquet | 2.400 | `data/raw/finance_csv.parquet` |
| Ibovespa (yfinance) | API REST → Parquet | ~740 | `data/raw/ibovespa.parquet` |
| Orçamento empresarial | Excel → Parquet | 144 | `data/raw/orcamento.parquet` |

**Próximo:** `02_transform.ipynb` — Limpeza, padronização e detecção de anomalias.